# 08 Optimize Induced Knowledge Graph


## 1. Imports and Configuration

The current graph is loaded from `data/knowledge_graph`. Validation data is used for optimization and candidate selection. Test data is loaded only for size/context checks and should not drive filtering decisions.

In [ ]:
from __future__ import annotations

import json
import math
import re
from collections import Counter
from itertools import product
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

KG_DIR = PROJECT_ROOT / "data" / "knowledge_graph"
NODES_PATH = KG_DIR / "nodes.csv"
EDGES_PATH = KG_DIR / "edges.csv"
EXPANDED_EDGES_PATH = KG_DIR / "edges_expanded.csv"

TRAIN_PATH = PROJECT_ROOT / "data" / "sentence_no_context" / "train_clean.csv"
VAL_PATH = PROJECT_ROOT / "data" / "sentence_no_context" / "val_clean.csv"
TEST_PATH = PROJECT_ROOT / "data" / "sentence_no_context" / "test_clean.csv"

OUTPUT_FILES = {
    "high_precision_graph": {
        "edges": KG_DIR / "edges_high_precision.csv",
        "mapping": KG_DIR / "mappings_high_precision.json",
    },
    "balanced_graph": {
        "edges": KG_DIR / "edges_balanced.csv",
        "mapping": KG_DIR / "mappings_balanced.json",
    },
    "high_coverage_graph": {
        "edges": KG_DIR / "edges_high_coverage.csv",
        "mapping": KG_DIR / "mappings_high_coverage.json",
    },
}

RANDOM_SEED = 42
PREVIEW_MAX_TERMS = 5
pd.set_option("display.max_colwidth", 180)
np.random.seed(RANDOM_SEED)

print(f"Project root: {PROJECT_ROOT}")
print(f"Knowledge graph directory: {KG_DIR.relative_to(PROJECT_ROOT)}")



## 2. Load Graph and Dataset Files

This section prints graph size, columns, relation distribution, and frequency/support statistics.

In [ ]:
GRAPH_EDGE_INPUT_PATH = EXPANDED_EDGES_PATH if EXPANDED_EDGES_PATH.exists() else EDGES_PATH

for path in [NODES_PATH, GRAPH_EDGE_INPUT_PATH, TRAIN_PATH, VAL_PATH, TEST_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

nodes_df = pd.read_csv(NODES_PATH)
edges_df = pd.read_csv(GRAPH_EDGE_INPUT_PATH)
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

required_text_columns = ["complex", "simple"]
for split_name, df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    missing = [column for column in required_text_columns if column not in df.columns]
    if missing:
        raise ValueError(f"{split_name} is missing columns: {missing}")
    for column in required_text_columns:
        df[column] = df[column].fillna("").astype(str).str.strip()

print(f"Nodes: {len(nodes_df):,}")
print(f"Edges: {len(edges_df):,}")
print(f"Graph edge input: {GRAPH_EDGE_INPUT_PATH.relative_to(PROJECT_ROOT)}")
print(f"Node columns: {list(nodes_df.columns)}")
print(f"Edge columns: {list(edges_df.columns)}")

if "relation" in edges_df.columns:
    display(edges_df["relation"].value_counts(dropna=False).rename_axis("relation").reset_index(name="count"))

stats_columns = [column for column in ["frequency", "supporting_sentence_pairs"] if column in edges_df.columns]
if stats_columns:
    display(edges_df[stats_columns].describe().T)

print(f"Train rows: {len(train_df):,}")
print(f"Validation rows: {len(val_df):,}")
print(f"Test rows: {len(test_df):,}  # not used for tuning")


## 3. Inspect Current Mappings



In [ ]:
frequency_column = "frequency" if "frequency" in edges_df.columns else None
support_column = "supporting_sentence_pairs" if "supporting_sentence_pairs" in edges_df.columns else None
sort_columns = [column for column in [frequency_column, support_column] if column]

mapping_columns = [column for column in [
    "source", "target", "frequency", "supporting_sentence_pairs", "example_complex", "example_simple"
] if column in edges_df.columns]

if sort_columns:
    top_mappings_df = edges_df.sort_values(sort_columns, ascending=False).head(50)
    bottom_mappings_df = edges_df.sort_values(sort_columns, ascending=True).head(50)
else:
    top_mappings_df = edges_df.head(50)
    bottom_mappings_df = edges_df.tail(50)

print("Top 50 mappings by frequency/support")
display(top_mappings_df[mapping_columns])

print("Bottom 50 mappings by frequency/support")
display(bottom_mappings_df[mapping_columns])

In [ ]:
MALFORMED_SUFFIXES = ("useds", "dissimilars", "preventions")
SUSPICIOUS_TARGET_PATTERNS = [
    r"\bthere\b",
    r"\bthe number\b",
    r"\blimited number\b",
    r"\bdebilitating side\b",
    r"\blarge loop excision\b",
    r"\bother\b.*\buseds\b",
]

def normalize_text(text: Any) -> str:
    text = str(text).lower().replace("_", " ").strip()
    text = re.sub(r"[^a-z0-9\s-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def looks_malformed_target(target: Any) -> bool:
    target_norm = normalize_text(target)
    if any(target_norm.endswith(suffix) for suffix in MALFORMED_SUFFIXES):
        return True
    return any(re.search(pattern, target_norm) for pattern in SUSPICIOUS_TARGET_PATTERNS)

malformed_df = edges_df[edges_df["target"].map(looks_malformed_target)].copy()
print(f"Mappings with malformed or suspicious targets: {len(malformed_df):,}")
display(malformed_df[mapping_columns])

manual_check_terms = [
    "incidence", "morbidity", "adverse", "other comparison", "other comparisons",
    "clinically heterogeneous", "prophylaxis", "prophylaxi"
]
manual_check_df = edges_df[edges_df["source"].map(lambda value: normalize_text(value) in manual_check_terms)].copy()
print("Suspicious examples requested for inspection")
display(manual_check_df[mapping_columns])

## 4. Graph Quality Scoring



In [ ]:
GENERIC_SOURCE_TERMS = {
    "risk", "rate", "effect", "effects", "quality", "delivery", "comparison", "comparisons",
    "other comparison", "other comparisons", "clinical", "adverse", "outcome", "outcomes",
    "intervention", "interventions", "incidence", "proportion", "proportions",
}

GENERIC_TARGET_TERMS = {
    "number", "numbers", "effect", "effects", "method", "methods", "treatment", "treatments",
    "risk", "rate", "rates", "there", "other", "different", "used", "useds",
}

BAD_TARGET_PATTERNS = [
    r"\bdebilitating side\b",
    r"\blimited number\b",
    r"\bthe number\b",
    r"\bthere\b",
    r"\buseds\b",
    r"\bdissimilars\b",
    r"\bpreventions\b",
]

NON_INFORMATIVE_PHRASES = {
    "there was", "there were", "we found", "we included", "this review", "this study",
    "other comparison", "other comparisons",
}

BIOMEDICAL_MARKERS = {
    "mortality", "morbidity", "adverse event", "myocardial", "infarction", "pulmonary",
    "cardiac", "renal", "corticosteroid", "steroid", "analgesia", "glucose", "lmwh",
    "cerclage", "nephrectomy", "progestin", "progesterone", "thyrotropin", "exacerbation",
    "prevalence", "efficacy", "hypothyroidism", "hyperthyroidism", "antibody", "antibiotic",
    "fracture", "transfusion", "pregnancy", "comorbidity", "comorbidities", "ductus",
}

READABLE_TARGET_HINTS = {
    "death", "side effect", "side effects", "heart attack", "lung", "blood sugar", "steroid",
    "steroids", "illness", "pain relief", "frequency", "percentage", "low-quality", "high-quality",
    "heart", "attack", "attacks", "surgery", "hormone", "treatment", "treatments",
}

def token_set(text: Any) -> set[str]:
    return set(normalize_text(text).split())

def overlap_ratio(source: Any, target: Any) -> float:
    source_tokens = token_set(source)
    target_tokens = token_set(target)
    if not source_tokens or not target_tokens:
        return 0.0
    return len(source_tokens & target_tokens) / max(len(source_tokens | target_tokens), 1)

def biomedical_source_score(source: Any) -> float:
    source_norm = normalize_text(source)
    tokens = source_norm.split()
    score = 0.0
    if any(marker in source_norm for marker in BIOMEDICAL_MARKERS):
        score += 2.0
    if any(len(token) >= 9 for token in tokens):
        score += 0.6
    if len(tokens) >= 2:
        score += 0.5
    if re.search(r"\b[a-z]{2,6}\b", source_norm) and source_norm in {"lmwh", "bmmnc", "rhtsh"}:
        score += 1.5
    return score

def target_readability_score(target: Any) -> float:
    target_norm = normalize_text(target)
    tokens = target_norm.split()
    if not tokens:
        return -2.0
    score = 0.0
    if target_norm in READABLE_TARGET_HINTS:
        score += 1.2
    if all(len(token) <= 10 for token in tokens):
        score += 0.5
    if len(tokens) <= 4:
        score += 0.4
    if any(re.search(pattern, target_norm) for pattern in BAD_TARGET_PATTERNS):
        score -= 2.5
    return score

def score_mapping(source: Any, target: Any, frequency: int | float | None = None, support: int | float | None = None) -> float:
    source_norm = normalize_text(source)
    target_norm = normalize_text(target)
    source_tokens = source_norm.split()
    target_tokens = target_norm.split()

    score = 0.0
    freq_value = 1 if frequency is None or pd.isna(frequency) else max(float(frequency), 1.0)
    support_value = freq_value if support is None or pd.isna(support) else max(float(support), 1.0)
    score += min(math.log1p(freq_value), 2.5)
    score += 0.3 * min(math.log1p(support_value), 2.0)

    score += biomedical_source_score(source_norm)
    score += target_readability_score(target_norm)

    if 1 <= len(source_tokens) <= 4:
        score += 0.3
    if 1 <= len(target_tokens) <= 5:
        score += 0.2
    if len(source_norm) < 4 or len(target_norm) < 3:
        score -= 1.5

    if source_norm in GENERIC_SOURCE_TERMS:
        score -= 1.2
    if target_norm in GENERIC_TARGET_TERMS:
        score -= 1.0
    if source_norm in NON_INFORMATIVE_PHRASES or target_norm in NON_INFORMATIVE_PHRASES:
        score -= 2.0
    if overlap_ratio(source_norm, target_norm) > 0.65:
        score -= 1.0
    if any(target_norm.endswith(suffix) for suffix in MALFORMED_SUFFIXES):
        score -= 2.5

    return round(score, 4)

scored_edges_df = edges_df.copy()
scored_edges_df["source_norm"] = scored_edges_df["source"].map(normalize_text)
scored_edges_df["target_norm"] = scored_edges_df["target"].map(normalize_text)
scored_edges_df["quality_score"] = scored_edges_df.apply(
    lambda row: score_mapping(
        row["source"],
        row["target"],
        row.get("frequency", 1),
        row.get("supporting_sentence_pairs", row.get("frequency", 1)),
    ),
    axis=1,
)
scored_edges_df["is_generic_source"] = scored_edges_df["source_norm"].isin(GENERIC_SOURCE_TERMS)
scored_edges_df["has_bad_target"] = scored_edges_df["target_norm"].map(
    lambda value: any(re.search(pattern, value) for pattern in BAD_TARGET_PATTERNS)
)

display(scored_edges_df.sort_values("quality_score", ascending=False)[mapping_columns + ["quality_score"]].head(30))
display(scored_edges_df.sort_values("quality_score", ascending=True)[mapping_columns + ["quality_score", "is_generic_source", "has_bad_target"]].head(30))

## 5. Configurable Filtering



In [ ]:
QUALITY_THRESHOLDS = {
    "low": 1.0,
    "medium": 2.2,
    "high": 3.2,
}

def clean_target(target: Any) -> str:
    target_norm = normalize_text(target)
    replacements = {
        "useds": "used",
        "dissimilars": "dissimilar",
        "preventions": "prevention",
    }
    for old, new in replacements.items():
        target_norm = re.sub(rf"\b{re.escape(old)}\b", new, target_norm)
    return target_norm

def filter_edges(
    edges: pd.DataFrame,
    min_frequency: int = 1,
    quality_level: str = "medium",
    generic_handling: str = "remove_generic_sources",
    target_cleaning: str = "on",
    max_mappings_per_source: int = 1,
) -> tuple[pd.DataFrame, dict[str, str]]:
    filtered = edges.copy()
    if "frequency" not in filtered.columns:
        filtered["frequency"] = 1
    if "supporting_sentence_pairs" not in filtered.columns:
        filtered["supporting_sentence_pairs"] = filtered["frequency"]

    filtered["source_norm"] = filtered["source"].map(normalize_text)
    filtered["target_norm"] = filtered["target"].map(normalize_text)
    if target_cleaning == "on":
        filtered["target_norm"] = filtered["target_norm"].map(clean_target)

    filtered["quality_score"] = filtered.apply(
        lambda row: score_mapping(row["source_norm"], row["target_norm"], row["frequency"], row["supporting_sentence_pairs"]),
        axis=1,
    )
    filtered["is_generic_source"] = filtered["source_norm"].isin(GENERIC_SOURCE_TERMS)
    filtered["has_bad_target"] = filtered["target_norm"].map(lambda value: any(re.search(pattern, value) for pattern in BAD_TARGET_PATTERNS))

    threshold = QUALITY_THRESHOLDS[quality_level]
    filtered = filtered[filtered["frequency"].ge(min_frequency)].copy()
    filtered = filtered[filtered["quality_score"].ge(threshold)].copy()
    if generic_handling == "remove_generic_sources":
        filtered = filtered[~filtered["is_generic_source"]].copy()
    if target_cleaning == "on":
        filtered = filtered[~filtered["has_bad_target"]].copy()

    filtered = filtered.sort_values(
        ["source_norm", "quality_score", "frequency", "supporting_sentence_pairs"],
        ascending=[True, False, False, False],
    )
    filtered = filtered.groupby("source_norm", as_index=False, group_keys=False).head(max_mappings_per_source).copy()

    best_for_prompt = filtered.sort_values(
        ["source_norm", "quality_score", "frequency", "supporting_sentence_pairs"],
        ascending=[True, False, False, False],
    ).drop_duplicates("source_norm", keep="first")
    mapping = dict(zip(best_for_prompt["source_norm"], best_for_prompt["target_norm"], strict=False))
    return filtered.reset_index(drop=True), mapping

config_rows = []
for min_frequency, quality_level, generic_handling, target_cleaning, max_mappings in product(
    [1, 2, 3],
    ["low", "medium", "high"],
    ["allow_all", "remove_generic_sources"],
    ["off", "on"],
    [1, 2],
):
    filtered_edges, mapping = filter_edges(
        scored_edges_df,
        min_frequency=min_frequency,
        quality_level=quality_level,
        generic_handling=generic_handling,
        target_cleaning=target_cleaning,
        max_mappings_per_source=max_mappings,
    )
    config_rows.append({
        "config_id": f"freq{min_frequency}_{quality_level}_{generic_handling}_{target_cleaning}_max{max_mappings}",
        "min_frequency": min_frequency,
        "quality_level": quality_level,
        "generic_handling": generic_handling,
        "target_cleaning": target_cleaning,
        "max_mappings_per_source": max_mappings,
        "retained_edges": len(filtered_edges),
        "prompt_mappings": len(mapping),
        "filtered_edges": filtered_edges,
        "mapping": mapping,
    })

configs_df = pd.DataFrame(config_rows)
display(configs_df.drop(columns=["filtered_edges", "mapping"]).sort_values(["prompt_mappings", "retained_edges"], ascending=False).head(20))

## 6. Coverage Evaluation on Validation Set



In [ ]:
def singularize_token(token: str) -> str:
    if len(token) > 4 and token.endswith("ies"):
        return token[:-3] + "y"
    if len(token) > 3 and token.endswith("s") and not token.endswith("ss"):
        return token[:-1]
    return token

def singularize_phrase(phrase: str) -> str:
    return " ".join(singularize_token(token) for token in normalize_text(phrase).split())

def plural_variants(term: str) -> set[str]:
    normalized = normalize_text(term)
    singular = singularize_phrase(normalized)
    variants = {normalized, singular}
    tokens = singular.split()
    if tokens:
        last = tokens[-1]
        plural_last = last[:-1] + "ies" if last.endswith("y") else last + "s"
        variants.add(" ".join(tokens[:-1] + [plural_last]))
    return {variant for variant in variants if variant}

def current_matcher(sentence: str, mapping: dict[str, str], max_terms: int = 5) -> list[tuple[str, str]]:
    sentence_norm = normalize_text(sentence)
    matches = []
    occupied_spans: list[tuple[int, int]] = []
    for term, simple_term in sorted(mapping.items(), key=lambda item: len(item[0]), reverse=True):
        for variant in sorted(plural_variants(term), key=len, reverse=True):
            pattern = rf"(?<![a-z0-9]){re.escape(variant)}(?![a-z0-9])"
            match = re.search(pattern, sentence_norm)
            if not match:
                continue
            span = match.span()
            overlaps = any(not (span[1] <= old_start or span[0] >= old_end) for old_start, old_end in occupied_spans)
            if overlaps:
                continue
            output_simple = simple_term
            if match.group(0).endswith("s") and not output_simple.endswith("s") and len(output_simple.split()) <= 3:
                output_simple = output_simple + "s"
            matches.append((match.group(0), output_simple))
            occupied_spans.append(span)
            break
        if len(matches) >= max_terms:
            break
    return matches

def fuzzy_phrase_match(sentence_norm: str, term: str) -> re.Match[str] | None:
    # Conservative fuzzy mode: only for 2+ token terms, allowing punctuation/space variation between tokens.
    tokens = normalize_text(term).split()
    if len(tokens) < 2:
        return None
    flexible = r"[\s-]+".join(re.escape(token) for token in tokens)
    return re.search(rf"(?<![a-z0-9]){flexible}(?![a-z0-9])", sentence_norm)

def improved_matcher(sentence: str, mapping: dict[str, str], max_terms: int = 5, use_fuzzy_for_multiword: bool = True) -> list[tuple[str, str]]:
    sentence_norm = normalize_text(sentence)
    normalized_sentence_tokens = [singularize_token(token) for token in sentence_norm.split()]
    singular_sentence = " ".join(normalized_sentence_tokens)
    matches = []
    occupied_spans: list[tuple[int, int]] = []
    for term, simple_term in sorted(mapping.items(), key=lambda item: (len(item[0].split()), len(item[0])), reverse=True):
        candidate_matches = []
        for variant in sorted(plural_variants(term), key=len, reverse=True):
            for surface in [sentence_norm, singular_sentence]:
                pattern = rf"(?<![a-z0-9]){re.escape(variant)}(?![a-z0-9])"
                match = re.search(pattern, surface)
                if match:
                    candidate_matches.append(match)
                    break
        if use_fuzzy_for_multiword and not candidate_matches:
            fuzzy_match = fuzzy_phrase_match(sentence_norm, term)
            if fuzzy_match:
                candidate_matches.append(fuzzy_match)
        if not candidate_matches:
            continue
        match = candidate_matches[0]
        span = match.span()
        if any(not (span[1] <= old_start or span[0] >= old_end) for old_start, old_end in occupied_spans):
            continue
        matches.append((match.group(0), simple_term))
        occupied_spans.append(span)
        if len(matches) >= max_terms:
            break
    return matches

def evaluate_coverage(df: pd.DataFrame, mapping: dict[str, str], matcher, max_terms: int = 5) -> dict[str, Any]:
    detected = df["complex"].fillna("").map(lambda sentence: matcher(sentence, mapping, max_terms=max_terms))
    counts = detected.map(len)
    term_counter = Counter()
    for pairs in detected:
        for source, target in pairs:
            term_counter[f"{source}->{target}"] += 1
    return {
        "examples_with_terms": int(counts.gt(0).sum()),
        "coverage_pct": round(100 * counts.gt(0).mean(), 2),
        "avg_terms_per_example": round(float(counts.mean()), 3),
        "top_detected_terms": term_counter.most_common(30),
    }

coverage_rows = []
for row in configs_df.itertuples(index=False):
    current_cov = evaluate_coverage(val_df, row.mapping, current_matcher, max_terms=5)
    improved_cov = evaluate_coverage(val_df, row.mapping, improved_matcher, max_terms=5)
    coverage_rows.append({
        "config_id": row.config_id,
        "min_frequency": row.min_frequency,
        "quality_level": row.quality_level,
        "generic_handling": row.generic_handling,
        "target_cleaning": row.target_cleaning,
        "max_mappings_per_source": row.max_mappings_per_source,
        "prompt_mappings": row.prompt_mappings,
        "current_coverage_pct": current_cov["coverage_pct"],
        "current_avg_terms": current_cov["avg_terms_per_example"],
        "improved_coverage_pct": improved_cov["coverage_pct"],
        "improved_avg_terms": improved_cov["avg_terms_per_example"],
        "top_detected_terms": improved_cov["top_detected_terms"],
    })

coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df.sort_values(["improved_coverage_pct", "prompt_mappings"], ascending=False).head(20))

## 7. Mapping Quality Reports



In [ ]:
def get_config(config_id: str) -> pd.Series:
    matches = configs_df[configs_df["config_id"].eq(config_id)]
    if len(matches) == 0:
        raise KeyError(f"Unknown config_id: {config_id}")
    return matches.iloc[0]

def report_config(config_id: str) -> None:
    row = get_config(config_id)
    filtered_edges = row["filtered_edges"]
    retained_pairs = set(zip(filtered_edges["source_norm"], filtered_edges["target_norm"], strict=False))
    all_pairs = set(zip(scored_edges_df["source_norm"], scored_edges_df["target_norm"], strict=False))
    removed_pairs = all_pairs - retained_pairs
    removed_df = scored_edges_df[scored_edges_df.apply(lambda item: (item["source_norm"], item["target_norm"]) in removed_pairs, axis=1)].copy()

    print(f"Config: {config_id}")
    print(f"Retained edges: {len(filtered_edges):,}")
    print(f"Prompt mappings: {len(row['mapping']):,}")
    cov = evaluate_coverage(val_df, row["mapping"], improved_matcher, max_terms=5)
    print(f"Validation coverage: {cov['examples_with_terms']:,} / {len(val_df):,} = {cov['coverage_pct']}%")
    print(f"Average terms per validation example: {cov['avg_terms_per_example']}")

    print("Top 30 retained mappings")
    display(filtered_edges.sort_values(["quality_score", "frequency"], ascending=False)[mapping_columns + ["quality_score"]].head(30))

    print("Top 30 removed mappings by original frequency/support")
    display(removed_df.sort_values(["frequency", "supporting_sentence_pairs", "quality_score"], ascending=False)[mapping_columns + ["quality_score", "is_generic_source", "has_bad_target"]].head(30))

    print("Removed noisy mappings")
    noisy_removed = removed_df[removed_df["is_generic_source"] | removed_df["has_bad_target"] | removed_df["quality_score"].lt(QUALITY_THRESHOLDS["medium"])]
    display(noisy_removed.sort_values("quality_score").head(30)[mapping_columns + ["quality_score", "is_generic_source", "has_bad_target"]])

# Choose one representative config for an initial report. Adjust config_id after reviewing coverage_df.
candidate_report_id = coverage_df.sort_values(["improved_coverage_pct", "prompt_mappings"], ascending=False).iloc[0]["config_id"]
report_config(candidate_report_id)

## 8. Select and Save Candidate Graphs

We export three graph versions:

- `high_precision_graph`: fewer, cleaner mappings.
- `balanced_graph`: moderate filtering and reasonable coverage.
- `high_coverage_graph`: broader coverage while removing malformed targets.

The selected configs are heuristic starting points. Use validation generation in a later notebook to choose the final graph for Knowledge-Enhanced BioBART.

In [ ]:
def select_candidate_config(kind: str) -> str:
    df = coverage_df.copy()
    if kind == "high_precision_graph":
        subset = df[
            df["quality_level"].eq("high")
            & df["generic_handling"].eq("remove_generic_sources")
            & df["target_cleaning"].eq("on")
            & df["min_frequency"].ge(2)
        ].copy()
        if subset.empty:
            subset = df[df["quality_level"].eq("high")].copy()
        return subset.sort_values(["improved_coverage_pct", "prompt_mappings"], ascending=False).iloc[0]["config_id"]

    if kind == "balanced_graph":
        subset = df[
            df["quality_level"].eq("medium")
            & df["generic_handling"].eq("remove_generic_sources")
            & df["target_cleaning"].eq("on")
            & df["min_frequency"].isin([1, 2])
        ].copy()
        if subset.empty:
            subset = df[df["quality_level"].eq("medium")].copy()
        return subset.sort_values(["improved_coverage_pct", "prompt_mappings"], ascending=False).iloc[0]["config_id"]

    if kind == "high_coverage_graph":
        subset = df[
            df["quality_level"].isin(["low", "medium"])
            & df["target_cleaning"].eq("on")
            & df["min_frequency"].eq(1)
        ].copy()
        return subset.sort_values(["improved_coverage_pct", "prompt_mappings"], ascending=False).iloc[0]["config_id"]

    raise ValueError(f"Unknown candidate kind: {kind}")

selected_candidates = {kind: select_candidate_config(kind) for kind in OUTPUT_FILES}
display(pd.DataFrame([{"candidate": key, "config_id": value} for key, value in selected_candidates.items()]))

saved_rows = []
for candidate_name, config_id in selected_candidates.items():
    row = get_config(config_id)
    filtered_edges = row["filtered_edges"].copy()
    mapping = row["mapping"]
    edge_path = OUTPUT_FILES[candidate_name]["edges"]
    mapping_path = OUTPUT_FILES[candidate_name]["mapping"]

    filtered_edges.to_csv(edge_path, index=False)
    with open(mapping_path, "w", encoding="utf-8") as file:
        json.dump(mapping, file, ensure_ascii=False, indent=2, sort_keys=True)

    cov = evaluate_coverage(val_df, mapping, improved_matcher, max_terms=5)
    saved_rows.append({
        "candidate": candidate_name,
        "config_id": config_id,
        "edges": len(filtered_edges),
        "mappings": len(mapping),
        "validation_coverage_pct": cov["coverage_pct"],
        "validation_avg_terms": cov["avg_terms_per_example"],
        "edge_path": str(edge_path.relative_to(PROJECT_ROOT)),
        "mapping_path": str(mapping_path.relative_to(PROJECT_ROOT)),
    })

saved_candidates_df = pd.DataFrame(saved_rows)
display(saved_candidates_df)

## 9. Prompt Preparation Preview



In [ ]:
PROMPT_TEMPLATE = """You are an expert in biomedical text simplification.

Rewrite the biomedical sentence for a general audience.

Rules:
- Preserve the original meaning.
- Use clear and simple language.
- Replace medical, scientific, or technical terms with simpler alternatives whenever possible.
- Remove unnecessary statistical details unless they are essential for understanding the main finding.
- Do not add information that is not present in the original sentence.
- Output exactly one simplified sentence.

Sentence:
{complex_sentence}

Simplified sentence:"""


def build_prompt(complex_sentence: str) -> str:
    return PROMPT_TEMPLATE.format(complex_sentence=str(complex_sentence).strip())


def build_knowledge_prompt(sentence: str, term_pairs: list[tuple[str, str]]) -> str:
    sentence = str(sentence).strip()
    if not term_pairs:
        return build_prompt(sentence)

    mapping_lines = "\n".join(f"- {source} = {target}" for source, target in term_pairs[:PREVIEW_MAX_TERMS])
    return f"""You are an expert in biomedical text simplification.

Rewrite the biomedical sentence for a general audience.

Rules:
- Preserve the original meaning.
- Use clear and simple language.
- Replace medical, scientific, or technical terms with simpler alternatives whenever possible.
- Remove unnecessary statistical details unless they are essential for understanding the main finding.
- Do not add information that is not present in the original sentence.
- Output exactly one simplified sentence.

Use these term simplifications if relevant:
{mapping_lines}

Sentence:
{sentence}

Simplified sentence:"""

def preview_candidate_prompts(candidate_name: str, n: int = 20) -> pd.DataFrame:
    config_id = selected_candidates[candidate_name]
    mapping = get_config(config_id)["mapping"]
    preview = val_df.copy()
    detected = preview["complex"].map(lambda sentence: improved_matcher(sentence, mapping, max_terms=5))
    preview["detected_mappings"] = detected.map(lambda pairs: "; ".join(f"{source}={target}" for source, target in pairs))
    preview["detected_count"] = detected.map(len)
    preview["knowledge_prompt_preview"] = [
        build_knowledge_prompt(sentence, pairs)
        for sentence, pairs in zip(preview["complex"].tolist(), detected.tolist(), strict=False)
    ]
    sample = preview[preview["detected_count"].gt(0)].sample(
        n=min(n, int(preview["detected_count"].gt(0).sum())),
        random_state=RANDOM_SEED,
    )
    return sample[["complex", "simple", "detected_mappings", "knowledge_prompt_preview"]]

for candidate_name in selected_candidates:
    print(f"Candidate: {candidate_name}")
    display(preview_candidate_prompts(candidate_name, n=20))


